In [ ]:
library(dplyr)
library(lme4)

indir <- "/path/to/nature_study/RNA_export_for_stats"
outdir <- "/path/to/nature_study/RNA_composition_models"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

prog_counts <- read.delim(
  file.path(indir, "progenitor_donor_counts_for_composition.tsv"),
  check.names = FALSE
)

prog_counts$condition <- factor(prog_counts$condition, levels = c("disomy", "Ts21"))

prog_counts

# binomial GLM
fit_binom <- glm(
  cbind(n_B, n_prog - n_B) ~ condition,
  data = prog_counts,
  family = binomial()
)

summary(fit_binom)

# quasibinomial GLM for possible overdispersion
fit_quasi <- glm(
  cbind(n_B, n_prog - n_B) ~ condition,
  data = prog_counts,
  family = quasibinomial()
)

summary(fit_quasi)

# odds ratio
coef_tab <- summary(fit_quasi)$coefficients
beta <- coef(fit_quasi)["conditionTs21"]
OR <- exp(beta)

cat("Ts21 odds ratio for B_primed_prog fraction:", OR, "\n")

# predicted fractions by donor and condition
prog_counts$pred_binom <- predict(fit_binom, type = "response")
prog_counts$pred_quasi <- predict(fit_quasi, type = "response")

write.table(
  prog_counts,
  file = file.path(outdir, "B_primed_prog_composition_binomial_predictions.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

saveRDS(
  list(fit_binom = fit_binom, fit_quasi = fit_quasi, data = prog_counts),
  file = file.path(outdir, "B_primed_prog_composition_binomial_models.rds")
)

In [ ]:
indir <- "/path/to/nature_study/RNA_export_for_stats"
outdir <- "/path/to/nature_study/RNA_composition_models"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

meta_prog <- read.delim(
  file.path(indir, "progenitor_cell_metadata_for_composition.tsv"),
  check.names = FALSE
)

meta_prog$condition <- factor(meta_prog$condition, levels = c("disomy", "Ts21"))
meta_prog$donor <- factor(meta_prog$donor)
meta_prog$is_B_primed_prog <- as.integer(meta_prog$is_B_primed_prog)

table(meta_prog$condition)
table(meta_prog$donor)
table(meta_prog$is_B_primed_prog)

fit_glmer <- glmer(
  is_B_primed_prog ~ condition + (1 | donor),
  data = meta_prog,
  family = binomial(),
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

summary(fit_glmer)

beta <- fixef(fit_glmer)["conditionTs21"]
OR <- exp(beta)
cat("Ts21 mixed-model odds ratio for B_primed_prog:", OR, "\n")

saveRDS(
  list(fit_glmer = fit_glmer),
  file = file.path(outdir, "B_primed_prog_composition_glmer_model.rds")
)

In [ ]:
prog_counts <- read.delim(
  "/path/to/nature_study/RNA_export_for_stats/progenitor_donor_counts_for_composition.tsv",
  check.names = FALSE
)

prog_counts$condition <- factor(prog_counts$condition, levels = c("disomy", "Ts21"))

obs_diff <- with(prog_counts, mean(frac_B[condition == "Ts21"]) - mean(frac_B[condition == "disomy"]))

set.seed(1)
B <- 100000
perm_diffs <- numeric(B)

for (i in seq_len(B)) {
  perm_cond <- sample(prog_counts$condition)
  perm_diffs[i] <- mean(prog_counts$frac_B[perm_cond == "Ts21"]) -
    mean(prog_counts$frac_B[perm_cond == "disomy"])
}

p_two_sided <- mean(abs(perm_diffs) >= abs(obs_diff))

cat("Observed difference:", obs_diff, "\n")
cat("Permutation p-value:", p_two_sided, "\n")

In [ ]:
summary(fit_quasi)
summary(fit_glmer)
prog_counts